In [8]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_e59b447ec3e8c16c2c33682236cfd6fe"

In [2]:
import kagglehub

path = kagglehub.competition_download("home-credit-default-risk")
print(path)

C:\Users\DESK0059-\.cache\kagglehub\competitions\home-credit-default-risk


C:\Users\DESK0059-\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
print(os.listdir(path))

['application_test.csv', 'application_train.csv', 'bureau.csv', 'bureau_balance.csv', 'credit_card_balance.csv', 'HomeCredit_columns_description.csv', 'installments_payments.csv', 'POS_CASH_balance.csv', 'previous_application.csv', 'sample_submission.csv']


### Item 1: Understand what each table and row represents

In [9]:
import pandas as pd 

app_train = pd.read_csv(os.path.join(path, "application_train.csv"))
print(app_train.shape)
app_train.head()
app_train.columns.value_counts()

(307511, 122)


SK_ID_CURR                    1
TARGET                        1
NAME_CONTRACT_TYPE            1
CODE_GENDER                   1
FLAG_OWN_CAR                  1
                             ..
AMT_REQ_CREDIT_BUREAU_DAY     1
AMT_REQ_CREDIT_BUREAU_WEEK    1
AMT_REQ_CREDIT_BUREAU_MON     1
AMT_REQ_CREDIT_BUREAU_QRT     1
AMT_REQ_CREDIT_BUREAU_YEAR    1
Name: count, Length: 122, dtype: int64

### Item 2 — How the Tables Connect

- `SK_ID_CURR` = the main key, identifies one loan application (present in almost every table)
- `application_train` (SK_ID_CURR) is the central table — everything else links back to it
- `bureau` connects via `SK_ID_CURR` → one applicant can have multiple past credits (with other banks)
- `bureau_balance` connects one level deeper via `SK_ID_BUREAU` → monthly status of each bureau credit
- `previous_application` connects via `SK_ID_CURR` → one applicant can have multiple past Home Credit loans, each with its own `SK_ID_PREV`
- `POS_CASH_balance`, `credit_card_balance`, `installments_payments` connect via `SK_ID_PREV` → monthly/payment-level history for each previous loan

**Relationship type:** one-to-many at every level (one applicant → many bureau credits → many monthly records)

**For Day 3:** only using `application_train.csv` — other tables noted here for future feature engineering, not joined yet.

### Item 3: Check Shapes, Columns, and Data Types

In [11]:
app_train.info()
app_train.isnull().sum().sort_values(ascending=False).head(15)
app_train.head()

<class 'pandas.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), str(16)
memory usage: 286.2 MB


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
